In [6]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [7]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
#warnings.filterwarnings('ignore')

In [8]:
start = dt.datetime(2019,5,25)
start = start.replace(hour=0, minute=0, second=0, microsecond=0)
#start = start - dt.timedelta(hours = 5, minutes = 30)
end = dt.datetime(2019,6,9)
end = end.replace(hour=0, minute=0, second=0, microsecond=0)
#end = end - dt.timedelta(hours = 5, minutes = 30)
print(start,end,end-start)

2019-05-25 00:00:00 2019-06-09 00:00:00 15 days, 0:00:00


In [9]:
query = (
    f"""SELECT
    * FROM 
    `hitwicketsuperstars.analytics_190927423.new_user_reference`"""
)
new_user_reference = client.query(query).to_dataframe()

In [12]:
query = (
    f"""SELECT
                     event_name,
                     user_id as device_id,
                     event_timestamp,
                     params.key,
                     params.value.string_value,
                     device.mobile_os_hardware_model
                     FROM
                     `hitwicketsuperstars.analytics_190927423.events_2019*`,
                     UNNEST(event_params) AS params 
                     WHERE _TABLE_SUFFIX BETWEEN '0525'
                     AND '0609'
                     AND event_name = 'popup' and params.key = 'action'
                     AND (params.value.string_value = 'Clicked_Yes_to_PlayStore')"""
)
events = client.query(query).to_dataframe()
print(len(events))
events.head()

278


,event_name,device_id,event_timestamp,key,string_value,mobile_os_hardware_model
0,popup,85f1df1d02d205ede5928d86462fde55,1559751312228001,action,Clicked_Yes_to_PlayStore,Redmi 6
1,popup,a13999488b45ed69e8269f3f9c9c2edd,1559786046081005,action,Clicked_Yes_to_PlayStore,Lenovo K8 Note
2,popup,d81eaf4450451a779b76ebc05ddbdc4a,1559838385106000,action,Clicked_Yes_to_PlayStore,Redmi Note 5 Pro
3,popup,e08a4ce8f86cf22acc9f65e32380f25b,1559808261286005,action,Clicked_Yes_to_PlayStore,Lenovo X3a40
4,popup,273df8dfb74f4a8448559e2c21b40de7,1559799365463001,action,Clicked_Yes_to_PlayStore,A33f


In [ ]:
# query = (
#     f"""SELECT
#                      event_name,
#                      user_id as device_id,
#                      event_timestamp,
#                      params.key,
#                      params.value.string_value

#                      FROM
#                      `hitwicketsuperstars.analytics_190927423.events_2019*`,
#                      UNNEST(event_params) AS params 
#                      WHERE _TABLE_SUFFIX BETWEEN '0525'
#                      AND '0609'
#                      params.key = 'action'"""
# )
# events = client.query(query).to_dataframe()
# print(len(events))
# events.head()

In [18]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}},{"sign_up_details":1, "created_at":1,"login_details":1}):
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
df = pd.DataFrame(dic_flattened)
#df = df[df["sign_up_details_device_id"].isin(devices)]
users = df[["_id","created_at","sign_up_details_device_id","login_details_last_request_at"]]
users.columns = ["user_id","createtime","device_id","last_request"]
print(len(users))
users.head()

4511


,user_id,createtime,device_id,last_request
0,5ce893c173bf607b6be1afc4,2019-05-25 01:00:49.430,fb2ce9a6f1166b5a5377a1a30c8c9866,2019-06-05 02:10:15.622
1,5ce898db12305c7b4a3065b9,2019-05-25 01:22:35.557,ba63bf3125d15fad921166d35e04cf4d,2019-05-29 05:29:02.291
2,5ce89b5e67847c1b7ca97294,2019-05-25 01:33:18.618,fa5ab462604dfe0ad1dccc676a962cc2,2019-05-25 01:36:19.295
3,5ce89e337a6eba7b449b207d,2019-05-25 01:45:23.104,debb35332abd5c68902dda64bf8a67e3,NaT
4,5ce89e3b9e0a5c4bc44c42c1,2019-05-25 01:45:31.643,debb35332abd5c68902dda64bf8a67e3,NaT


In [27]:
yes_users = pd.merge(events[['device_id','event_timestamp']],users[['device_id','last_request']],on='device_id',how = 'inner')
yes_users['event_timestamp'] = pd.to_datetime(yes_users['event_timestamp'], unit='us')
len(yes_users)

210

In [29]:
absolute_yes_users = yes_users[yes_users['last_request']>yes_users['event_timestamp']]
len(absolute_yes_users)

181

In [33]:
percentage_of_users = len(absolute_yes_users)/len(yes_users)
percentage_of_users*100

86.19047619047619